<a href="https://colab.research.google.com/github/Juliusp36/GinyuForce2/blob/main/notebooks/02_baselines.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1 Part A: Load the MovieLens ratings

In [4]:
import os, urllib.request, zipfile
import numpy as np
import pandas as pd

if not os.path.exists('ml-100k'):
  urllib.request.urlretrieve(
      'https://files.grouplens.org/datasets/movielens/ml-100k.zip',
      'ml-100k.zip')
  zipfile.ZipFile('ml-100k.zip').extractall()

ratings = pd.read_csv('ml-100k/u.data', sep='\t', names=['user', 'item', 'rating', 'time'])

print(ratings.head())
print('Number of ratings:', len(ratings))

   user  item  rating       time
0   196   242       3  881250949
1   186   302       3  891717742
2    22   377       1  878887116
3   244    51       2  880606923
4   166   346       1  886397596
Number of ratings: 100000


The code above downloads the movielens 100k dataset from the internet and loads the rating into a table with the columns of users, items, rating, time.

2 Part B: Make an honest train/test split

In [15]:
def split_ratings(ratings, test_frac=0.10, seed=0):

  rng = np.random.default_rng(seed)
  is_test = np.zeros(len(ratings), dtype=bool)

  user_left = ratings.groupby('user').size().to_dict()
  item_left = ratings.groupby('item').size().to_dict()

  for pos in rng.permutation(len(ratings)):
    row = ratings.iloc[pos]
    user, item = row['user'], row['item']

    if (rng.random() < test_frac
        and user_left[user] > 1
        and item_left[item] > 1):
      is_test[pos] = True
    user_left[user] -= 1
    item_left[item] -= 1
  return ratings.iloc[~is_test].copy(), ratings.iloc[is_test].copy()

train, test = split_ratings(ratings)
print('train:', len(train), 'test:', len(test))
print('test fraction:', round(len(test) / len(ratings), 3))


train: 90246 test: 9754
test fraction: 0.098


This code splits the ratings into train and set. It vists every rating in random order. For each one, it moves the trating to the test set only if it meet all the statements in the if statement.

3. Part C: Build the training matrix

In [17]:
N_USERS, N_ITEMS = 943, 1682

def to_matrix(df):
  R = np.full((N_USERS, N_ITEMS), np.nan)
  R[df['user'].to_numpy() -1,
    df['item'].to_numpy() -1] = df['rating'].to_numpy()
  return R

R = to_matrix(train)
known = np.isnan(R)

print('Matrix shape:', R.shape)
print('Known training rating:', known.sum())


Matrix shape: (943, 1682)
Known training rating: 1495880
